# 02 - Ablations and results table

Run **00_build_library.ipynb first**. Runs the ablation grid over several
seeds and prints a mean +/- std summary plus a LaTeX table you can paste
into the paper. Each ablation isolates one design choice flagged in review.

In [ ]:
import os, sys, copy
sys.path.insert(0, os.path.abspath('ppg_breakhis'))
import numpy as np, pandas as pd, torch
from config import Config
from data.breakhis import make_datasets
from run_ablations import ABLATIONS, run_one   # reused, no CLI

ABLATIONS  # inspect what will run

## 1. Configure

In [ ]:
DATA_ROOT = '/content/BreakHis_v1'   # <-- EDIT THIS
SEEDS = [0, 1, 2]

base = Config()
base.data_root = DATA_ROOT
base.epochs = 30
base.device = 'cuda' if torch.cuda.is_available() else 'cpu'

# build the datasets ONCE so ablation differences come from the method,
# not from a different patient split
datasets = make_datasets(base)
print({k: len(v) for k, v in datasets.items()})

## 2. Run the grid
This trains many models - reduce `SEEDS` or `base.epochs` for a quick pass.

In [ ]:
raw = {name: [] for name in ABLATIONS}
for name, ov in ABLATIONS.items():
    for seed in SEEDS:
        m = run_one(base, ov, datasets, seed)
        raw[name].append(m)
        print(f"{name:20s} seed={seed} acc={m['accuracy']:.3f} "
              f"auc={m['auc']:.3f} ece={m['ece']:.3f}")

## 3. Summary (mean +/- std over seeds)

In [ ]:
def summarize(raw, keys=('accuracy', 'auc', 'f1', 'ece', 'nll')):
    rows = {}
    for name, runs in raw.items():
        rows[name] = {k: (np.nanmean([r[k] for r in runs]),
                          np.nanstd([r[k] for r in runs])) for k in keys}
    return rows

summary = summarize(raw)
disp = pd.DataFrame({
    name: {k: f"{mu:.3f} \u00b1 {sd:.3f}" for k, (mu, sd) in row.items()}
    for name, row in summary.items()
}).T
disp.rename_axis('ablation')

## 4. LaTeX table for the paper
Bold-friendly plain table; add `\textbf{}` around your best rows by hand.

In [ ]:
latex = disp[['accuracy', 'auc', 'ece']].to_latex(
    caption='Ablation on BreakHis (200x), mean $\\pm$ std over seeds.',
    label='tab:ablation', column_format='lccc')
print(latex)
with open('ablation_table.tex', 'w') as fh:
    fh.write(latex)
print('saved ablation_table.tex')